In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [ ]:

%pip install --quiet geopandas fiona

%pip install rasterio

%pip install pystac



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.2/208.2 kB 5.3 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

import rasterio
import os
import csv

import json
import xml.etree.ElementTree as ET
import datetime
import requests
from io import BytesIO

from rasterio.warp import transform_bounds

from matplotlib.colors import ListedColormap, Normalize
from shapely.geometry import shape, mapping, MultiPolygon, Polygon,box
from IPython.display import Image, display


import numpy as np
import pystac
from pystac.extensions.table import TableExtension
from pystac import CatalogType
from pystac import Asset, MediaType
from pystac.extensions.classification import ClassificationExtension, Classification
from pystac.extensions.raster import RasterExtension,RasterBand
from pystac.extensions.projection import ProjectionExtension


### **Used ijson python package for reading the large GEOJSON files and get the metadata **

In [ ]:
pip install ijson

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.0/149.0 kB 5.0 MB/s eta 0:00:00


In [ ]:
import datetime
from pystac import Item
import pandas as pd
import geopandas as gpd
import os
import ijson
import json
from shapely.geometry import shape, mapping

In [ ]:
# #geojson_filepath = '/content/drive/MyDrive/Pan_india/pan_india_drainage_lines.geojson'
# geojson_filepath = '/content/drive/MyDrive/Pan_india/Microwatershed_boundries_v2.geojson'

# geojson_filepath = "/content/drive/MyDrive/PAN_india_vector_layerstest.csv"

#geojson_filepath = "/content/drive/MyDrive/PAN_india_layers_updated.csv"

geojson_filepath = "/content/drive/MyDrive/PAN_india_vector_layers_mapping.csv"

SUB_COLLECTIONS = {}

STAC_SAVE_DIR = "/content/drive/MyDrive/STAC_spec_PANIndia_test/PanIndiaCatalogs"
ROOT_CATALOG_HREF = os.path.join(STAC_SAVE_DIR, "catalog.json")
PANINDIA_COLLECTION_ID = "PanIndiaCatalogs"


In [ ]:
df = pd.read_csv(geojson_filepath)

print("DataFrame loaded successfully")
print(df.head())

DataFrame loaded successfully
             collection_name display_name                   layer_name  \
0  Administrative boundaries        State      state_boundaries_vector   
1  Administrative boundaries     District   district_boundaries_vector   
2  Administrative boundaries       Tehsil     tehsil_boundaries_vector   
3  Administrative boundaries    Panchayat  panchayat_boundaries_vector   
4  Administrative boundaries      Village    village_boundaries_vector   

                                          asset link  \
0  /content/drive/MyDrive/Pan_india/State_pan_ind...   
1  /content/drive/MyDrive/Pan_india/District_pan_...   
2  /content/drive/MyDrive/Pan_india/SOI_tehsil.ge...   
3  /content/drive/MyDrive/Pan_india/Panchayat_pan...   
4  /content/drive/MyDrive/Pan_india/Village_pan_i...   

                                          drive_link  
0  https://drive.google.com/file/d/1TwH079-5uSKYJ...  
1  https://drive.google.com/file/d/1h7c0-nfjAKrth...  
2  https://drive.google

In [ ]:
try:

    COLUMN_DESC_DF = pd.read_csv('/content/drive/MyDrive/column_description_9dec.csv')
    print("Column descriptions loaded successfully.")
except FileNotFoundError:
    print("WARNING: 'column_descriptions.csv' not found. Table extension will be empty.")
    COLUMN_DESC_DF = pd.DataFrame({'layer_name': [], 'column_name': [], 'column_name_description': []})

Column descriptions loaded successfully.


In [ ]:
def create_root_and_collection():
    root_collection_href = os.path.join(STAC_SAVE_DIR, "collection.json")

    if os.path.exists(root_collection_href):
        print(f"Loading Existing Root Collection: {root_collection_href}")

        root_collection = pystac.Collection.from_file(root_collection_href)
    else:
        print(f"Creating New Root Collection at: {STAC_SAVE_DIR}")
        if not os.path.exists(STAC_SAVE_DIR):
            os.makedirs(STAC_SAVE_DIR)

        extent = pystac.Extent(
    spatial=pystac.SpatialExtent(
        bboxes=[[68.0, 6.0, 97.0, 37.0]]
    ),
    temporal=pystac.TemporalExtent(
        intervals=[[
            datetime.datetime(2017, 1, 1),
            datetime.datetime(2024, 6, 30)
        ]]
    )
)

        root_collection = pystac.Collection(
            id='PanIndiaCatalogs',
            title='Pan India Spatio Temporal Asset Catalog',
            description='This spatio temporal asset catalog contains all data layers of CoRE Stack (https://core-stack.org/) generated at Pan India level.',
            extent=extent,
            license='CC-BY-4.0'
        )


    root_collection.set_self_href(root_collection_href)
    return root_collection, root_collection

In [ ]:
def create_sub_collection(title, parent_collection):

    title_str = str(title) if title is not None else "unknown"
    collection_id = title_str.lower().replace(' ', '-').replace(':', '').replace('/', '-')

    description = f"STAC collection for {title_str} of India."

    if collection_id not in SUB_COLLECTIONS:
        print(f"Creating new Sub-Collection: {title_str}")
        sub_collection = pystac.Collection(
            id=collection_id,
            title=title_str,
            description=description,
            extent=parent_collection.extent,
            license=parent_collection.license,
            providers=parent_collection.providers
        )


        SUB_COLLECTIONS[collection_id] = sub_collection
    return SUB_COLLECTIONS[collection_id]


In [ ]:

def get_feature_geometry(filepath):
    ext = os.path.splitext(filepath)[1].lower()

    if ext in [".geojson", ".json"]:
        with open(filepath, 'r', encoding="utf-8") as f:
            for item in ijson.items(f, 'features.item', use_float=True):
                geom = item.get("geometry")
                shapely_geom = shape(geom)
                minx, miny, maxx, maxy = shapely_geom.bounds
                return geom, [minx, miny, maxx, maxy]


    if ext == ".shp":

        for enc in ["utf-8", "latin-1", "cp1252"]:
            try:
                gdf = gpd.read_file(filepath, encoding=enc)
                break
            except UnicodeDecodeError:
                continue


        first_geom = gdf.iloc[0].geometry
        geom = mapping(first_geom)
        bbox = gdf.total_bounds.tolist()

        return geom, bbox

    raise ValueError(f"Unsupported vector file format: {filepath}")


def get_first_feature_properties(filepath):
    ext = os.path.splitext(filepath)[1].lower()


    if ext in [".geojson", ".json"]:
        with open(filepath, 'r', encoding="utf-8") as f:
            for item in ijson.items(f, 'features.item.properties', use_float=True):
                properties_dict = {
                    key: {"value": value, "type": type(value).__name__}
                    for key, value in item.items()
                }
                break

        return {"properties": properties_dict}


    if ext == ".shp":

        for enc in ["utf-8", "latin-1", "cp1252"]:
            try:
                gdf = gpd.read_file(filepath, encoding=enc)
                break
            except UnicodeDecodeError:
                continue

        props_raw = gdf.iloc[0].to_dict()


        properties_dict = {
            key: {"value": value, "type": type(value).__name__}
            for key, value in props_raw.items()
        }

        return {"properties": properties_dict}

    raise ValueError(f"Unsupported vector file format: {filepath}")


In [ ]:

def load_layer_descriptions(csv_path="/content/drive/MyDrive/layer_descriptions.csv"):
    desc_map = {}
    with open(csv_path, "r") as f:
        reader = csv.DictReader(f)
        for row in reader:

            norm_key = (
                row["layer_name"]
                .lower()
                .replace(" ", "-")
                .replace(":", "")
                .replace("/", "-")
                .strip()
            )
            desc_map[norm_key] = row["layer_description"].strip()
    return desc_map

LAYER_DESCRIPTIONS = load_layer_descriptions("/content/drive/MyDrive/layer_descriptions.csv")


In [ ]:
def generate_vector_stac(layer_name, file_path, column_desc_df, display_name=None, drive_link=None):
    try:
        print(f"Processing vector layer: {layer_name}")


        item_id = (
            layer_name.lower()
            .replace(" ", "_")
            .replace("/", "_")
            .replace("-", "_")
            .strip()
        )

        def clean_key(s):
            return str(s).lower().replace(" ", "").replace("-", "").replace("_", "").strip()

        target_key = clean_key(layer_name)
        item_description = layer_name


        for k, v in LAYER_DESCRIPTIONS.items():
            if clean_key(k) == target_key:
                item_description = v
                break


        footprint, bbox = get_feature_geometry(file_path)
        properties_dict = get_first_feature_properties(file_path)["properties"]


        vector_gdf_dtypes = pd.DataFrame(
            [(key, value['type']) for key, value in properties_dict.items()],
            columns=['column_name', 'column_dtype']
        )


        item_properties = {
            "title": display_name if display_name else layer_name,
            "description": item_description
        }

        vector_item = pystac.Item(
            id=item_id,
            geometry=footprint,
            bbox=bbox,
            datetime=datetime.datetime.now(datetime.timezone.utc),
            properties=item_properties
        )


        proj_ext = ProjectionExtension.ext(vector_item, add_if_missing=True)
        proj_ext.epsg = 4326


        temp_col_df = column_desc_df.copy()
        temp_col_df['match_key'] = temp_col_df["layer_name"].apply(clean_key)

        vector_desc_filtered_df = temp_col_df[temp_col_df["match_key"] == target_key]

        vector_merged_df = vector_gdf_dtypes.merge(
            vector_desc_filtered_df[['column_name', 'column_name_description']],
            on='column_name',
            how='left'
        ).fillna('')

        vector_merged_df.rename(
            columns={'column_name_description': 'column_description'},
            inplace=True
        )

        print(
            f"Schema merged for {layer_name}. "
            f"Found {len(vector_merged_df[vector_merged_df['column_description'] != ''])} column descriptions."
        )


        table_ext = TableExtension.ext(vector_item, add_if_missing=True)
        table_ext.columns = [
            {
                "name": row['column_name'],
                "type": str(row['column_dtype']),
                "description": row['column_description']
            }
            for idx, row in vector_merged_df.iterrows()
        ]


        if drive_link:
            vector_item.add_asset(
                "drive-link",
                pystac.Asset(
                    href=drive_link,
                    media_type=pystac.MediaType.TEXT,
                    roles=["source"],
                    title="Asset File"
                )
            )

        return vector_item

    except Exception as e:
        print(f"Error generating STAC Item for {layer_name}: {e}")
        return None

In [ ]:
def run_stac_generation(df, root_collection, COLUMN_DESC_DF):
    generated_items_count = 0
    sub_collections_map = {}

    for index, row in df.iterrows():
        category_name = row['collection_name']
        layer_id = row['layer_name']
        file_path = row['asset link']
        drive_link = row['drive_link']
        layer_display_name = row['display_name']

        coll_id = category_name.lower().replace(' ', '_').replace('-', '_')

        if coll_id not in sub_collections_map:

            target_sub = root_collection.get_child(coll_id)

            if not target_sub:
                print(f"NEW CATEGORY FOUND: Adding {coll_id} to collection.json")
                target_sub = pystac.Collection(
                    id=coll_id,
                    title=category_name,
                    description=f"STAC collection for {category_name} of India.",
                    extent=root_collection.extent,
                    license='CC-BY-4.0'
                )

                root_collection.add_child(target_sub)

                root_collection.normalize_hrefs(STAC_SAVE_DIR)
                root_collection.save(catalog_type=pystac.CatalogType.SELF_CONTAINED)

            sub_collections_map[coll_id] = target_sub

        target_sub = sub_collections_map[coll_id]

        if not target_sub.get_item(layer_id):
            if os.path.exists(file_path):
                print(f"Adding Item: {layer_id}")

                vector_item = generate_vector_stac(
                    layer_name=layer_id,
                    file_path=file_path,
                    column_desc_df=COLUMN_DESC_DF,
                    display_name=layer_display_name,
                    drive_link=drive_link
                )

                if vector_item:
                    vector_item.id = layer_id
                    target_sub.add_item(vector_item)
                    generated_items_count += 1

                    if generated_items_count % 5 == 0:
                        root_collection.normalize_hrefs(STAC_SAVE_DIR)
                        root_collection.save(catalog_type=pystac.CatalogType.SELF_CONTAINED)
        else:
            print(f"Item {layer_id} already exists in {coll_id}. Skipping.")

    root_collection.normalize_hrefs(STAC_SAVE_DIR)
    root_collection.save(catalog_type=pystac.CatalogType.SELF_CONTAINED)
    print(f"New items added: {generated_items_count}")


In [ ]:
def save_catalog(root_catalog, count):

    if count >= 0:

        root_catalog.normalize_hrefs(STAC_SAVE_DIR)

        root_catalog.save(catalog_type=pystac.CatalogType.SELF_CONTAINED)
        print(f"STAC catalog saved.")
        print(f"Total items generated:{count}")
    else:
        print("No items were generated.")

In [ ]:

# root_catalog, panindia_collection = get_or_create_root_catalog()

# run_stac_generation(df, panindia_collection, root_catalog, COLUMN_DESC_DF)

root_collection, _ = create_root_and_collection()

run_stac_generation(df, root_collection, COLUMN_DESC_DF)

Loading Existing Root Collection: /content/drive/MyDrive/STAC_spec_PANIndia_test/PanIndiaCatalogs/collection.json
NEW CATEGORY FOUND: Adding administrative_boundaries to collection.json
Adding Item: state_boundaries_vector
Processing vector layer: state_boundaries_vector
Schema merged for state_boundaries_vector. Found 1 column descriptions.
Adding Item: district_boundaries_vector
Processing vector layer: district_boundaries_vector
Schema merged for district_boundaries_vector. Found 8 column descriptions.
Adding Item: tehsil_boundaries_vector
Processing vector layer: tehsil_boundaries_vector
Schema merged for tehsil_boundaries_vector. Found 5 column descriptions.
Adding Item: panchayat_boundaries_vector
Processing vector layer: panchayat_boundaries_vector
Schema merged for panchayat_boundaries_vector. Found 21 column descriptions.
Adding Item: village_boundaries_vector
Processing vector layer: village_boundaries_vector
Schema merged for village_boundaries_vector. Found 24 column descri